# 1. EDA and data quality
Load a SQL Server extract (or a Parquet/CSV snapshot), validate the canonical schema, and inspect coverage, nulls, signs, codes, and activity.

In [ ]:
# %pip install -r ../requirements.txt
import os, sys
from pathlib import Path
import polars as pl
import plotly.express as px
sys.path.insert(0, str(Path.cwd().parents[1] / 'src'))
from transaction_analysis import load_extract, load_sql_server, profile_transactions

DATA_PATH = os.getenv('DATA_PATH', '')
if DATA_PATH:
    transactions = load_extract(DATA_PATH)
else:
    # Set SQL_CONNECTION_STRING and replace the source table/query as needed.
    from pathlib import Path
    query = Path('../sql/01_customer_aggregation.sql').read_text()
    transactions = load_sql_server(query, connection_string=os.environ['SQL_CONNECTION_STRING'])
print(profile_transactions(transactions))

In [ ]:
# Quality and coverage tables
transactions.describe(), transactions.group_by('movement_type').len().sort('len', descending=True)

In [ ]:
monthly = (transactions.with_columns(pl.col('transaction_date').dt.truncate('1mo').alias('period'))
           .group_by('period').agg(pl.len().alias('transactions'), pl.col('customer_id').n_unique().alias('customers'))
           .sort('period'))
px.line(monthly.to_pandas(), x='period', y=['transactions','customers'], title='Transaction and customer coverage').show()

Review invalid movement codes, duplicate IDs, missing descriptions/banks, and whether negative amounts agree with movement type before proceeding.